# 🧠 EpiConnectome Tutorial
## A Step-by-Step Guide to EEG Connectivity Analysis of Epileptic Seizures

**Project:** EpiConnectome — Brainhack School 2026  
**Dataset:** Siena Scalp EEG Database (PhysioNet)  
**GitHub:** https://github.com/duhmariya/EEG_Project

---

### What you will learn in this notebook

| Step | Topic |
|---|---|
| 0 | Import libraries and set up reproducibility |
| 1 | Configure paths and find all seizure files |
| 2 | Understand the pipeline (what happens to each file) |
| 3 | Run the full pipeline on the entire Siena dataset |
| 4 | Explore your connectivity results |
| 5 | Visualize the group-level connectivity atlas |

### Prerequisites
- Basic Python knowledge
- No prior EEG experience required!

### Setup
```bash
conda env create -f environment.yml
conda activate epiconnectome
jupyter notebook
```

### Dataset Download
```bash
wget -r -N -c -np https://physionet.org/files/siena-scalp-eeg/1.0.0/
```
Then run `01_siena_to_txt_converter.py` to convert EDF → TXT before using this notebook.

---
## 📦 Step 0 — Import Libraries & Reproducibility Settings

We always set a fixed **random seed** before anything else. This ensures that every time you run this notebook, you get **identical results** — a core principle of reproducible science.

| Library | Purpose |
|---|---|
| `mne` | EEG loading, preprocessing, source analysis |
| `mne_connectivity` | wPLI and other connectivity metrics |
| `numpy` | Numerical arrays and math |
| `pandas` | Data tables and Excel export |
| `matplotlib / seaborn` | Visualization |
| `networkx` | Graph theory metrics |
| `glob` | Finding files on disk |

In [ ]:
import os
import glob
import random
import warnings
warnings.filterwarnings('ignore')

import mne
import mne_connectivity
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# ── Reproducibility — set this before anything else ───────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
mne.set_config('MNE_RANDOM_SEED', str(RANDOM_SEED))
mne.set_config('MNE_USE_NUMBA', 'false')
mne.set_log_level('WARNING')

print('✅ Libraries loaded')
print(f'   MNE:              v{mne.__version__}')
print(f'   MNE-connectivity: v{mne_connectivity.__version__}')
print(f'   NumPy:            v{np.__version__}')
print(f'   Pandas:           v{pd.__version__}')
print(f'   Random seed:      {RANDOM_SEED} (reproducibility guaranteed)')

---
## ⚙️ Step 1 — Configure Paths & Parameters

This is the **only cell you need to change** to run on your own machine.

Set `TXT_FOLDER` to where your converted Siena TXT files are, and `OUTPUT_FOLDER` to where you want results saved.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CHANGE THESE TWO PATHS FOR YOUR MACHINE
# ══════════════════════════════════════════════════════════════
TXT_FOLDER    = r"C:\Users\mariy\Desktop\Siena\siena-scalp-eeg-database-1.0.0\siena_txt_format"
OUTPUT_FOLDER = r"C:\Users\mariy\Desktop\Siena\Notebook_Results_Full"
# ══════════════════════════════════════════════════════════════

# ── Pipeline parameters ───────────────────────────────────────
SFREQ         = 512       # Original Siena sampling rate (Hz)
TARGET_SFREQ  = 1000      # Resampled rate (Hz)
EPOCH_LENGTH  = 10        # Epoch duration (seconds)
VIZ_THRESHOLD = 0.0       # Minimum wPLI to show in circle plots
NET_THRESHOLD = 0.5       # Minimum wPLI to count as an edge in graph metrics

# ── Standard 16-channel 10-20 montage ─────────────────────────
# These are the channels present in all Siena recordings
CHANNEL_NAMES = [
    'Fp1', 'Fp2',              # Prefrontal
    'F7',  'F3', 'Fz', 'F4', 'F8',  # Frontal
    'T3',  'C3', 'Cz', 'C4', 'T4',  # Central/Temporal
    'T5',  'P3', 'Pz', 'P4'   # Parietal/Temporal
]

# ── Frequency bands ───────────────────────────────────────────
# Each band captures different brain dynamics during seizures
FREQ_BANDS = {
    'theta': (6,  8),   # Hippocampal/memory networks
    'alpha': (8,  12),  # Inhibitory/thalamocortical networks
    'beta':  (12, 30),  # Cognitive/motor networks
}

# ── Find all TXT files ────────────────────────────────────────
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
txt_files = sorted(glob.glob(os.path.join(TXT_FOLDER, '**', '*.txt'), recursive=True))

print(f'✅ Configuration ready')
print(f'   Input folder:  {TXT_FOLDER}')
print(f'   Output folder: {OUTPUT_FOLDER}')
print(f'   Files found:   {len(txt_files)}')
print(f'   Freq bands:    {FREQ_BANDS}')
print()
print('Files to process:')
for i, f in enumerate(txt_files, 1):
    print(f'   {i:2d}. {os.path.basename(f)}')

---
## 📖 Step 2 — Understand the Pipeline

Before running, let's understand what happens to each file. This section explains the concepts — no code to run here.

### The Full Pipeline (per file)

```
TXT File (samples × channels)
       │
       ▼
  1. LOAD & TRIM
     Remove first/last 30s (boundary noise)
       │
       ▼
  2. MNE RAW OBJECT
     Assign channel names + electrode positions
       │
       ▼
  3. PREPROCESSING
     ├── Bandpass filter: 4–40 Hz
     ├── Resample: 512 → 1000 Hz
     ├── Common average reference
     ├── Bad channel detection (LOF) + interpolation
     └── ICA: remove eye + muscle artifacts
       │
       ▼
  4. EPOCHING
     Cut into non-overlapping 10-second windows
       │
       ▼
  5. wPLI CONNECTIVITY
     Compute for theta, alpha, beta bands
     Output: 16×16 matrix per band
       │
       ▼
  6. GRAPH THEORY METRICS
     Global efficiency, clustering, modularity
       │
       ▼
  7. SAVE OUTPUTS
     ├── Excel: connectivity matrices
     └── PNG: circle plots per band
```

### Key Concepts

**wPLI (Weighted Phase Lag Index)**  
Measures consistent phase relationships between channels. Values range 0–1. We use wPLI specifically because it is robust to volume conduction — the artifact where the same brain signal appears in multiple electrodes due to electrical spread through the skull.

**ICA (Independent Component Analysis)**  
Separates brain signals from artifacts. Eye blinks create huge deflections in frontal channels (Fp1, Fp2). Muscle tension creates high-frequency noise. ICA identifies these as separate components and removes them before connectivity is computed.

**Epoching**  
We cut the continuous recording into 10-second windows and compute wPLI across all windows. Averaging over multiple epochs gives a more stable, reliable connectivity estimate than using a single long segment.

---
## 🚀 Step 3 — Run the Full Pipeline

This cell processes **every file** in your TXT folder automatically.

For each file it will:
- Print progress as it goes
- Save a subfolder of results per subject
- Skip and continue if a file fails
- Build a summary table at the end

> ⏱️ **Expected time:** ~3–5 minutes per file (ICA is the slow step). For the full 47 seizures, plan for 3–4 hours. You can run overnight or process a subset first by changing `txt_files[:3]` to test on 3 files.

In [ ]:
# ── Helper functions ──────────────────────────────────────────

def load_txt(path):
    """Load Siena TXT file. Falls back to pandas if numpy fails."""
    try:
        data = np.loadtxt(path)
    except:
        data = pd.read_csv(path, sep='\s+', header=None).values
    if data.ndim == 2 and data.shape[1] > 16:
        data = data[:, :16]
    return data


def preprocess(eeg_data):
    """Full preprocessing pipeline: filter → resample → ref → bad channels → ICA."""
    # Trim 30s from each end
    trim = 30 * SFREQ
    if eeg_data.shape[0] > 2 * trim:
        eeg_data = eeg_data[trim:-trim, :]

    # Create MNE Raw
    info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=SFREQ, ch_types=['eeg'] * 16)
    raw  = mne.io.RawArray(eeg_data.T, info, verbose=False)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, match_alias=True, on_missing='ignore')

    # Filter + resample + reference
    raw.filter(l_freq=4, h_freq=40, verbose=False)
    raw.resample(TARGET_SFREQ, verbose=False)
    raw, _ = mne.set_eeg_reference(raw, projection=True, verbose=False)

    # Bad channel detection
    bad_channels, _ = mne.preprocessing.find_bad_channels_lof(
        raw, n_neighbors=8, threshold=1.5, return_scores=True)
    raw.info['bads'] = bad_channels
    if bad_channels:
        raw.interpolate_bads(reset_bads=True)

    # ICA
    ica = mne.preprocessing.ICA(
        n_components=15, random_state=RANDOM_SEED,
        method='infomax', max_iter='auto')
    ica.fit(inst=raw.copy().filter(4, 40, verbose=False), verbose=False)

    eog_ch = [ch for ch in ['Fp1', 'Fp2'] if ch in raw.ch_names]
    if eog_ch:
        eog_idx, _ = ica.find_bads_eog(inst=raw, ch_name=eog_ch,
                                         measure='correlation', threshold=0.5)
    else:
        eog_idx = []
    muscle_idx, _ = ica.find_bads_muscle(inst=raw, threshold=0.5)
    ica.exclude = list(set(eog_idx + muscle_idx))
    raw = ica.apply(raw.copy())

    return raw, bad_channels, ica.exclude


def make_epochs(raw):
    """Cut continuous raw into non-overlapping 10s epochs."""
    interval = TARGET_SFREQ * EPOCH_LENGTH
    events   = np.array([[i, 0, 1] for i in range(0, raw.n_times, interval)])
    return mne.Epochs(raw, events, event_id=1, tmin=0,
                      tmax=EPOCH_LENGTH - 1/TARGET_SFREQ,
                      baseline=None, preload=True, verbose=False)


def compute_wpli(epochs):
    """Compute wPLI for all frequency bands. Returns dict of 16x16 matrices."""
    fmin = [b[0] for b in FREQ_BANDS.values()]
    fmax = [b[1] for b in FREQ_BANDS.values()]
    con  = mne_connectivity.spectral_connectivity_epochs(
        epochs, method='wpli', mode='multitaper',
        fmin=fmin, fmax=fmax, faverage=True,
        sfreq=epochs.info['sfreq'], verbose=False)
    raw_matrix = con.get_data(output='dense')

    conn_dict = {}
    for idx, band in enumerate(FREQ_BANDS.keys()):
        m = raw_matrix[:, :, idx]
        m = m + m.T           # Symmetrize
        np.fill_diagonal(m, 0)
        conn_dict[band] = m
    return conn_dict


def compute_graph_metrics(con_matrix):
    """Build networkx graph and compute topology metrics."""
    G = nx.Graph()
    n = con_matrix.shape[0]
    for i in range(n):
        for j in range(i+1, n):
            if con_matrix[i, j] > NET_THRESHOLD:
                G.add_edge(i, j, weight=con_matrix[i, j])
    if G.number_of_edges() == 0:
        return {'n_edges': 0, 'density': 0, 'global_efficiency': 0,
                'avg_clustering': 0, 'modularity': float('nan')}
    metrics = {
        'n_edges':           G.number_of_edges(),
        'density':           nx.density(G),
        'global_efficiency': nx.global_efficiency(G),
        'avg_clustering':    nx.average_clustering(G, weight='weight'),
    }
    try:
        communities = nx.community.greedy_modularity_communities(G)
        metrics['modularity'] = nx.community.modularity(G, communities)
    except:
        metrics['modularity'] = float('nan')
    return metrics


def save_excel(conn_dict, graph_metrics, path):
    """Save connectivity matrices and graph metrics to Excel."""
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        for band, matrix in conn_dict.items():
            df = pd.DataFrame(matrix, index=CHANNEL_NAMES, columns=CHANNEL_NAMES)
            df.to_excel(writer, sheet_name=band.capitalize())
        df_metrics = pd.DataFrame(graph_metrics).T
        df_metrics.index.name = 'Band'
        df_metrics.to_excel(writer, sheet_name='Graph_Metrics')


def save_circle_plots(conn_dict, out_dir, file_stem):
    """Save one circle plot per frequency band."""
    n_ch = len(CHANNEL_NAMES)
    idx_row, idx_col = np.triu_indices(n_ch, k=1)
    indices = (idx_row, idx_col)

    for band, con_matrix in conn_dict.items():
        upper_tri = con_matrix[idx_row, idx_col]
        fig, _ = plot_connectivity_circle(
            upper_tri, CHANNEL_NAMES,
            indices=indices,
            vmin=VIZ_THRESHOLD,
            vmax=max(con_matrix.max(), 0.01),
            title=f'{file_stem} — {band.capitalize()} Band',
            colormap='hot', colorbar=True, show=False)
        out_path = os.path.join(out_dir, f'circle_{band}.png')
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)


print('✅ Helper functions defined — ready to run pipeline')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MAIN LOOP — processes all files
#  To test on a subset first, change txt_files to txt_files[:3]
# ══════════════════════════════════════════════════════════════

all_results  = []   # Collects summary row per file per band
success      = 0
failed_files = []

for file_idx, txt_path in enumerate(txt_files, 1):
    file_stem = os.path.splitext(os.path.basename(txt_path))[0]
    print(f'\n[{file_idx}/{len(txt_files)}] {file_stem}')
    print(f'{"-"*50}')

    try:
        # ── 1. Load ───────────────────────────────────────────
        print('  [1/5] Loading...', end=' ')
        eeg_data = load_txt(txt_path)
        print(f'shape {eeg_data.shape}')

        # ── 2. Preprocess ────────────────────────────────────
        print('  [2/5] Preprocessing...', end=' ')
        raw, bad_ch, excluded_ica = preprocess(eeg_data)
        print(f'bad={bad_ch if bad_ch else "none"}, ICA removed={len(excluded_ica)}')

        # ── 3. Epoch ─────────────────────────────────────────
        print('  [3/5] Epoching...', end=' ')
        epochs = make_epochs(raw)
        print(f'{len(epochs)} epochs × {EPOCH_LENGTH}s')

        # ── 4. wPLI ──────────────────────────────────────────
        print('  [4/5] Computing wPLI...', end=' ')
        conn_dict = compute_wpli(epochs)
        for band, m in conn_dict.items():
            mean_val = np.mean(m[m != 0])
            print(f'{band}={mean_val:.3f}', end='  ')
        print()

        # ── 5. Graph metrics ──────────────────────────────────
        graph_metrics = {}
        for band, matrix in conn_dict.items():
            graph_metrics[band] = compute_graph_metrics(matrix)

        # ── 6. Save outputs ───────────────────────────────────
        print('  [5/5] Saving...', end=' ')
        subj_out = os.path.join(OUTPUT_FOLDER, file_stem)
        os.makedirs(subj_out, exist_ok=True)

        excel_path = os.path.join(subj_out, f'{file_stem}_connectivity.xlsx')
        save_excel(conn_dict, graph_metrics, excel_path)
        save_circle_plots(conn_dict, subj_out, file_stem)
        print(f'→ {subj_out}')

        # ── 7. Store summary row ──────────────────────────────
        for band, matrix in conn_dict.items():
            gm = graph_metrics[band]
            all_results.append({
                'Subject':           file_stem,
                'Band':              band,
                'Mean_wPLI':         round(np.mean(matrix[matrix != 0]), 4),
                'Max_wPLI':          round(matrix.max(), 4),
                'N_Epochs':          len(epochs),
                'Bad_Channels':      len(bad_ch),
                'ICA_Excluded':      len(excluded_ica),
                'Global_Efficiency': round(gm.get('global_efficiency', 0), 4),
                'Avg_Clustering':    round(gm.get('avg_clustering', 0), 4),
                'Density':           round(gm.get('density', 0), 4),
                'Modularity':        round(gm.get('modularity', float('nan')), 4),
            })

        print(f'  ✅ Complete')
        success += 1

    except Exception as e:
        print(f'  ❌ Failed: {str(e)}')
        failed_files.append(file_stem)
        continue

# ── Save group summary ────────────────────────────────────────
print(f'\n{"="*60}')
print(f'COMPLETE — Success: {success}/{len(txt_files)}, Failed: {len(failed_files)}')
if failed_files:
    print(f'Failed files: {failed_files}')
print(f'{"="*60}')

df_summary = pd.DataFrame(all_results)
summary_path = os.path.join(OUTPUT_FOLDER, 'ALL_subjects_connectivity_atlas.xlsx')
df_summary.to_excel(summary_path, index=False)
print(f'\n📊 Atlas saved → {summary_path}')

---
## 🔍 Step 4 — Explore the Results

Now that the pipeline has run, let's look at what was produced.

The summary table has one row per subject per frequency band. We can use it to answer questions like:
- Which frequency band shows the strongest connectivity during seizures?
- Which subjects have the highest global efficiency?
- Is there a consistent pattern across all 47 seizures?

In [ ]:
# Load atlas (useful if you re-open the notebook after running)
atlas_path = os.path.join(OUTPUT_FOLDER, 'ALL_subjects_connectivity_atlas.xlsx')
df = pd.read_excel(atlas_path)

print(f'Atlas shape: {df.shape} (rows × columns)')
print(f'Subjects: {df["Subject"].nunique()}')
print(f'Bands: {df["Band"].unique()}')
print()
print('── Mean values per frequency band ──')
print(df.groupby('Band')[['Mean_wPLI','Max_wPLI','Global_Efficiency',
                           'Avg_Clustering','Density']].mean().round(3).to_string())
print()
print('── First 5 rows ──')
df.head()

---
## 📊 Step 5 — Visualize the Group-Level Atlas

### 5a. Mean wPLI per Band across all Subjects

This shows which frequency band has the strongest seizure connectivity on average.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics    = ['Mean_wPLI', 'Global_Efficiency', 'Avg_Clustering']
titles     = ['Mean wPLI\n(connectivity strength)', 
               'Global Efficiency\n(information flow)',
               'Avg Clustering\n(local connectivity)']
band_colors = {'theta': '#2196F3', 'alpha': '#4CAF50', 'beta': '#FF9800'}

for ax, metric, title in zip(axes, metrics, titles):
    band_means = df.groupby('Band')[metric].mean()
    band_stds  = df.groupby('Band')[metric].std()
    bands      = list(FREQ_BANDS.keys())
    colors     = [band_colors[b] for b in bands]

    bars = ax.bar(bands, [band_means[b] for b in bands],
                  yerr=[band_stds[b] for b in bands],
                  color=colors, alpha=0.8, edgecolor='black',
                  linewidth=0.8, capsize=5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency Band', fontsize=11)
    ax.set_ylabel(metric.replace('_', ' '), fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=11)

    # Annotate bars
    for bar, band in zip(bars, bands):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + band_stds[band] + 0.005,
                f'{band_means[band]:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Group-Level EEG Connectivity Atlas — Siena Seizure Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'group_connectivity_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: group_connectivity_summary.png')

### 5b. Subject-Level wPLI Heatmap

This heatmap shows each subject's mean wPLI per band — useful for spotting outliers or subject-specific patterns.

In [ ]:
pivot = df.pivot_table(index='Subject', columns='Band', values='Mean_wPLI')
pivot = pivot[list(FREQ_BANDS.keys())]  # Ensure band order

fig, ax = plt.subplots(figsize=(8, max(6, len(pivot) * 0.35)))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    annot=True, fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Mean wPLI'},
    ax=ax
)
ax.set_title('Mean wPLI per Subject per Band\n(Siena Seizure Dataset)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency Band', fontsize=11)
ax.set_ylabel('Subject', fontsize=11)
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'subject_band_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: subject_band_heatmap.png')

### 5c. Channel-Level Connectivity — Which Channels Are Hubs?

Load one subject's matrix and look at which channels have the highest connectivity strength during their seizure.

In [ ]:
# Load the first subject's alpha connectivity matrix as an example
first_subject = df['Subject'].iloc[0]
excel_path    = os.path.join(OUTPUT_FOLDER, first_subject,
                              f'{first_subject}_connectivity.xlsx')

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
colors = {'theta': '#2196F3', 'alpha': '#4CAF50', 'beta': '#FF9800'}

for ax, band in zip(axes, FREQ_BANDS.keys()):
    matrix = pd.read_excel(excel_path, sheet_name=band.capitalize(),
                            index_col=0).values
    # Channel strength = mean connection weight per channel
    channel_strength = np.mean(matrix, axis=1)

    bars = ax.bar(CHANNEL_NAMES, channel_strength,
                  color=colors[band], alpha=0.8,
                  edgecolor='black', linewidth=0.5)
    ax.axhline(y=0.3, color='red', linestyle='--',
               linewidth=1.5, label='Threshold = 0.3')
    ax.set_title(f'{band.capitalize()} Band\n({FREQ_BANDS[band][0]}–{FREQ_BANDS[band][1]} Hz)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Channel', fontsize=10)
    ax.set_ylabel('Mean wPLI Strength', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Channel Connectivity Strength — {first_subject}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, f'{first_subject}_channel_strength.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {first_subject}_channel_strength.png')

---
## ✅ Summary

You have now run the complete EpiConnectome pipeline on the full Siena dataset.

### What was produced

```
Notebook_Results/
├── ALL_subjects_connectivity_atlas.xlsx   ← Full connectivity atlas
├── group_connectivity_summary.png         ← Group-level bar charts
├── subject_band_heatmap.png               ← Subject × band heatmap
│
├── PN00_seizure1/
│   ├── PN00_seizure1_connectivity.xlsx    ← 16×16 matrices (theta/alpha/beta)
│   ├── circle_theta.png
│   ├── circle_alpha.png
│   └── circle_beta.png
│
├── PN01_seizure1/
│   └── ...
└── ...
```

### Key parameters used

| Parameter | Value |
|---|---|
| Channels | 16 (standard 10-20) |
| Filter | 4–40 Hz |
| Sampling rate | 512 → 1000 Hz |
| Epoch length | 10 seconds |
| Connectivity | wPLI (multitaper) |
| Bands | θ (6–8), α (8–12), β (12–30) Hz |
| ICA | 15 components, infomax |
| Random seed | 42 |

### Next steps

- 📓 Use `04_siencehisto.py` to batch-visualize all Excel connectivity files
- 🔬 Coming soon: Source-level analysis with dSPM + HCPMMP1 parcellation
- 🤖 Coming soon: GNN/GATv2 classification using connectivity matrices as input

---
### References

- Detti, P. (2020). Siena Scalp EEG Database. PhysioNet. https://doi.org/10.13026/5d4a-j060
- Gramfort et al. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience.*
- Vinck et al. (2011). An improved index of phase-synchronization for electrophysiological data. *NeuroImage.*

---
*EpiConnectome · Brainhack School 2026 · github.com/duhmariya/EEG_Project*